# TCGA-Eingabedaten: Klassen, Verteilungen und Varianz pro Downstream-Task

**Reine Datenexploration:** kein VAE, kein β, keine Posterioren und keine AUC-Schätzung.
Die Berechnungen stehen in `tcga_data_analysis.py`. Wir untersuchen die RNA- und
Methylierungsdateien auf ihrer **tatsächlich gespeicherten Skala**, nicht nach dem
AUTOENCODIX-Filter auf 2000 Features. Es erfolgt keine nachträgliche Harmonisierung.
In den RNA-Daten kommen auch negative Werte vor; sie werden nicht entfernt oder
stillschweigend logarithmiert. Ihr Ursprung muss bei biologischen Schlussfolgerungen
anhand der Quelldaten geklärt werden.

**Zwei verschiedene Fragen:**

1. Wo ist viel Varianz zwischen Patienten, unabhängig von Labels?
2. Wie viel davon liegt zwischen den Klassen eines bestimmten Downstream-Tasks?

Ein Feature kann stark variieren und trotzdem keine mittleren Klassenunterschiede
haben. Deshalb zeigen wir Gesamtvarianz und η² getrennt. η² ist eine deskriptive
Assoziation auf diesen Daten, kein kausaler Effekt und keine validierte Vorhersage.


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Image
from beta_vae_experiments.tcga_data_analysis import (
    FILES, task_groups, run_tcga_data_analysis, plot_variance_association, plot_task_boxplots,
)
ROOT = Path.cwd()  # Start Jupyter from the repository root.
DATA = ROOT/'data/raw/TCGAPanCancerAutoencodix'
OUTPUT = ROOT/'beta_vae_experiments/analysis_results/tcga_data'
OUTPUT.mkdir(parents=True, exist_ok=True)
RECOMPUTE = False
if RECOMPUTE or not (OUTPUT/'provenance.json').is_file():
    effects = run_tcga_data_analysis(DATA, OUTPUT)
else:
    effects = pd.concat([pd.read_parquet(OUTPUT/f'{modality}_feature_variance.parquet')
                         for modality in FILES], ignore_index=True)
clinical = pd.read_parquet(OUTPUT/'paired_clinical.parquet')
groups = task_groups(clinical)
counts = pd.read_csv(OUTPUT/'class_counts.csv')
cohorts = pd.read_csv(OUTPUT/'task_cohorts.csv')
moments = pd.read_csv(OUTPUT/'class_moments.csv')
print('Inputs:', DATA)
print('Outputs:', OUTPUT)


## 1. Welche Daten und Patienten liegen vor?

Die gemeinsame Kohorte ist die Schnittmenge der RNA- und Methylierungs-Sample-IDs.
Fehlende klinische Labels bleiben erhalten und werden pro Aufgabe ausgewiesen.
Es wird kein Train-/Validation-/Test-Split für diese Beschreibung verwendet.


In [ ]:
display(pd.read_csv(OUTPUT/'data_overview.csv'))
print('Paired molecular samples:', len(clinical))
print('Known cancer labels:', clinical.CANCER_TYPE_ACRONYM.notna().sum())
print('Cancer types:', clinical.CANCER_TYPE_ACRONYM.nunique())
print('Distinct patient IDs:', clinical.PATIENT_ID.nunique())
display(cohorts)
display(pd.read_csv(OUTPUT/'value_quality.csv'))


## 2. Downstream-Klassen und Klassenungleichgewicht

Krebsart und Krebsgruppe benutzen die komplette gepaarte Kohorte mit bekanntem
Label. BRCA-/UCEC-Subtypen sind auf die jeweilige Krebsart beschränkt. Für Sex
zeigen wir die gesamte Kohorte und zusätzlich einzelne Krebsarten aus Folie 42.
`n_outside_cohort` ist keine fehlende Annotation; `n_missing` zählt nur fehlende
Labels innerhalb der jeweiligen Task-Kohorte.


In [ ]:
fig,axes = plt.subplots(2,2,figsize=(15,11))
for ax,task in zip(axes.flat,['cancer_type','sex_all','brca_subtype','ucec_subtype']):
    frame=counts.loc[counts.task.eq(task)].sort_values('n')
    ax.barh(frame.group,frame.n,color='#2F6DB5')
    ax.set(title=task,xlabel='Patients with known label')
    ax.grid(axis='x',alpha=.2)
fig.tight_layout()
fig.savefig(OUTPUT/'downstream_class_counts.png',dpi=170)
plt.close(fig)
display(Image(filename=str(OUTPUT/'downstream_class_counts.png'),width=1100))


In [ ]:
sex_counts = pd.crosstab(clinical.CANCER_TYPE_ACRONYM,clinical.SEX).reindex(columns=['Female','Male'],fill_value=0)
sex_fraction = sex_counts.div(sex_counts.sum(axis=1),axis=0)
sex_counts.to_csv(OUTPUT/'sex_by_cancer_counts.csv')
fig,axes=plt.subplots(1,2,figsize=(12,11))
sns.heatmap(sex_counts,annot=True,fmt='d',cmap='Blues',ax=axes[0])
sns.heatmap(sex_fraction,annot=True,fmt='.2f',vmin=0,vmax=1,cmap='Blues',ax=axes[1])
axes[0].set_title('Known sex: counts by cancer type')
axes[1].set_title('Known sex: fractions by cancer type')
fig.tight_layout();fig.savefig(OUTPUT/'sex_by_cancer.png',dpi=170);plt.close(fig)
display(Image(filename=str(OUTPUT/'sex_by_cancer.png'),width=1000))


## 3. Wertebereiche und Alter nach Krebsart

Die Boxplots der Patientenmittelwerte beschreiben **Mittel über Gene eines
Patienten**. Sie sind keine Markerwerte und keine Expressionsverteilung eines
einzelnen Gens. Große Unterschiede zwischen Krebsarten können biologische und
technische Ursachen haben. Die RNA- und Methylierungsskalen werden getrennt gezeigt.
Boxplots blenden einzelne Ausreißerpunkte aus; die Daten werden nicht beschnitten.


In [ ]:
patient_metrics = clinical[['CANCER_TYPE_ACRONYM','AGE']].copy()
for modality in FILES:
    values=pd.read_csv(OUTPUT/f'{modality}_patient_summary.csv').set_index('sample_id')
    patient_metrics[modality+'_mean']=values['mean'].reindex(patient_metrics.index)
    patient_metrics[modality+'_std']=values['std'].reindex(patient_metrics.index)
    patient_metrics[modality+'_negative_fraction']=values.negative_fraction.reindex(patient_metrics.index)
order=clinical.CANCER_TYPE_ACRONYM.value_counts().index
fig,axes=plt.subplots(1,3,figsize=(16,11))
for ax,column in zip(axes,['RNA_mean','Methylation_mean','AGE']):
    sns.boxplot(data=patient_metrics,x=column,y='CANCER_TYPE_ACRONYM',order=order,
                showfliers=False,color='#77aadd',ax=ax)
    ax.set(title=column,ylabel='Cancer type')
fig.tight_layout();fig.savefig(OUTPUT/'patient_summary_and_age_boxplots.png',dpi=170);plt.close(fig)
display(Image(filename=str(OUTPUT/'patient_summary_and_age_boxplots.png'),width=1200))
display(clinical.groupby('CANCER_TYPE_ACRONYM').AGE.agg(['count','mean','std','median','min','max']).round(2))


## 4. Wo ist Varianz — und wo ist sie mit dem Task assoziiert?

Pro Feature und Task (jeweils nur Patienten mit gültigem Task-Label):

\[
V_{\mathrm{gesamt}}=V_{\mathrm{innerhalb}}+V_{\mathrm{zwischen}},\qquad
\eta^2=V_{\mathrm{zwischen}}/V_{\mathrm{gesamt}}.
\]

Die Zerlegung ist mit Patientenhäufigkeiten gewichtet und benutzt ddof=0.
Konstante Features erhalten kein definiertes η². Fehlende Featurewerte werden
nicht aufgefüllt; die Fallzahl `n` und die vertretenen Klassen stehen in der Tabelle.

**η² nahe 0:** kaum mittlere Klassenunterschiede, auch wenn die Varianz groß ist.
**η² nahe 1:** in dieser Kohorte liegen die Unterschiede überwiegend zwischen Klassen.
Das erkennt keine rein nichtlinearen oder ausschließlich varianzabhängigen
Klassenunterschiede. Viele kleine Klassen können η² erhöhen; Aufgaben sind deshalb
nicht allein anhand ihrer höchsten η²-Werte gegeneinander zu bewerten.


In [ ]:
plot_variance_association(effects,['cancer_type','sex_all','brca_subtype','ucec_subtype'],OUTPUT)
display(Image(filename=str(OUTPUT/'variance_vs_class_association.png'),width=1100))
ranking_columns=['task','modality','feature','n','n_classes','total_variance',
                 'within_variance','between_variance','eta_squared']
for task in ['cancer_type','sex_all','brca_subtype','ucec_subtype']:
    print(task,': highest descriptive between-class fractions')
    ranked=effects.loc[effects.task.eq(task)].sort_values('eta_squared',ascending=False)
    display(ranked.groupby('modality').head(5)[ranking_columns].round(5))


## 5. Einzelne Features: Klassen-Boxplots und Standardmomente

Die folgenden Features wurden explorativ auf denselben Daten ausgewählt:
pro Modalität das größte η² für die jeweilige Aufgabe. Gene sind mit ihren IDs
bezeichnet; es wird keine ungesicherte Gen-Namenszuordnung vorgenommen.
Vollständige Tabellen enthalten außerdem Beispiele mit hoher Gesamtvarianz.

Pro Klasse: n, fehlende Werte, Mittelwert, Stichprobenvarianz/-standardabweichung
(ddof=1), Median, Quartile, Schiefe und Excess-Kurtosis (Normalverteilung: 0).
Schiefe/Kurtosis bleiben bei zu kleinen oder konstanten Gruppen undefiniert.


### Krebsart


In [ ]:
plot_task_boxplots('cancer_type', effects, clinical, OUTPUT)
display(Image(filename=str(OUTPUT/'cancer_type_feature_boxplots.png'),width=1100))
chosen=effects.loc[effects.task.eq('cancer_type')].dropna(subset=['eta_squared']).sort_values('eta_squared',ascending=False).groupby('modality').head(1)
for item in chosen.itertuples():
    display(moments.loc[moments.task.eq('cancer_type') & moments.modality.eq(item.modality)
                        & moments.feature.astype(str).eq(str(item.feature))].round(4))


### Sex: gesamte PanCancer-Kohorte


In [ ]:
plot_task_boxplots('sex_all', effects, clinical, OUTPUT)
display(Image(filename=str(OUTPUT/'sex_all_feature_boxplots.png'),width=1100))
chosen=effects.loc[effects.task.eq('sex_all')].dropna(subset=['eta_squared']).sort_values('eta_squared',ascending=False).groupby('modality').head(1)
for item in chosen.itertuples():
    display(moments.loc[moments.task.eq('sex_all') & moments.modality.eq(item.modality)
                        & moments.feature.astype(str).eq(str(item.feature))].round(4))


### BRCA-Subtypen


In [ ]:
plot_task_boxplots('brca_subtype', effects, clinical, OUTPUT)
display(Image(filename=str(OUTPUT/'brca_subtype_feature_boxplots.png'),width=1100))
chosen=effects.loc[effects.task.eq('brca_subtype')].dropna(subset=['eta_squared']).sort_values('eta_squared',ascending=False).groupby('modality').head(1)
for item in chosen.itertuples():
    display(moments.loc[moments.task.eq('brca_subtype') & moments.modality.eq(item.modality)
                        & moments.feature.astype(str).eq(str(item.feature))].round(4))


### UCEC-Subtypen


In [ ]:
plot_task_boxplots('ucec_subtype', effects, clinical, OUTPUT)
display(Image(filename=str(OUTPUT/'ucec_subtype_feature_boxplots.png'),width=1100))
chosen=effects.loc[effects.task.eq('ucec_subtype')].dropna(subset=['eta_squared']).sort_values('eta_squared',ascending=False).groupby('modality').head(1)
for item in chosen.itertuples():
    display(moments.loc[moments.task.eq('ucec_subtype') & moments.modality.eq(item.modality)
                        & moments.feature.astype(str).eq(str(item.feature))].round(4))


## 6. Sex innerhalb von Krebsarten

Sex in der gesamten TCGA-Kohorte ist mit Krebsart und Organ konfundiert.
Deshalb betrachten wir COAD, LUAD, LUSC und READ zusätzlich getrennt. Dies ist
Datenexploration innerhalb der vier Krebsarten, keine Behauptung, dass diese
Kohorten alle Geschlechtsunterschiede erklären oder identische Effektstärken haben.


In [ ]:
sex_tasks=['sex_all','sex_COAD','sex_LUAD','sex_LUSC','sex_READ']
sex_top=effects.loc[effects.task.isin(sex_tasks)].dropna(subset=['eta_squared']).sort_values('eta_squared',ascending=False).groupby(['task','modality']).head(5)
display(sex_top[ranking_columns].round(5))
for task in ['sex_COAD','sex_LUAD']:
    plot_task_boxplots(task,effects,clinical,OUTPUT)
    display(Image(filename=str(OUTPUT/f'{task}_feature_boxplots.png'),width=1000))


## 7. Weitere Aufgaben und Grenzen

Auch Krebsgruppen, Histologie und Stadium stehen in den exportierten Tabellen.
Stadium über alle Krebsarten kann unterschiedliche klinische Konventionen und
Krebsartunterschiede vermischen; es ist keine einheitliche biologische Skala.

Die Analyse ist bewusst **labelbasiert und deskriptiv**. Hohe Gesamtvarianz ist
kein Beweis für Task-Information, hohes η² kein unabhängiger Marker-Nachweis.
Die ausgewählten Features sind nicht auf einem zurückgehaltenen Datensatz validiert.
Für spätere Vorhersageexperimente muss Feature-Auswahl ausschließlich auf Train
stattfinden. Hier werden weder die Trainingspipeline noch gespeicherte VAE-Ergebnisse verändert.


In [ ]:
print('Full per-feature variance tables: *_feature_variance.parquet')
print('Selected feature moments by class: class_moments.csv')
print('Selected feature rankings: top_features_by_task.csv')
print('Data/label coverage: data_overview.csv, value_quality.csv, task_cohorts.csv')
print('Output:',OUTPUT)
